<a href="https://colab.research.google.com/github/suasgn/s2/blob/main/final/final_proposed_model_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install deps

In [1]:
! pip install datasets evaluate transformers[torch] rouge-score nltk nusacrowd jsonlines indobenchmark-toolkit sentencepiece
# fix from https://github.com/indobenchmark/indobenchmark-toolkit/issues/3
! pip install accelerate==0.25.0
! pip install transformers==4.33.1
! pip install tokenizers==0.13.3
! pip install -U datasets
! pip install --no-cache-dir "git+https://github.com/suasgn/Top2Vec.git"

! apt install git-lfs

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.2/384.2 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 119.1 MB/s eta 0:00:00
   

## Set environment

In [2]:
from huggingface_hub import notebook_login, login
from google.colab import userdata
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

login(userdata.get("HF_TOKEN"))
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

Mounted at /content/drive


In [3]:
model_checkpoint = "indobenchmark/indobart-v2"

## Croscek dependensi

In [4]:
import accelerate
print(accelerate.__version__)

0.25.0


/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [5]:
import os, transformers
root = os.path.dirname(transformers.__file__)
print("transformers version:", transformers.__version__)
print("transformers path   :", root)
print("DETA file exists    :", os.path.exists(os.path.join(root, "models", "deta", "configuration_deta.py")))

transformers version: 4.33.1
transformers path   : /usr/local/lib/python3.11/dist-packages/transformers
DETA file exists    : True


## Kostum Tokenizer [inlined]
adapted supaya bisa jalan di google colab

In [6]:
import os
from shutil import copyfile
from typing import TYPE_CHECKING, Any, Dict, List, NamedTuple, Optional, Sequence, Tuple, Union
from transformers import PreTrainedTokenizer, BatchEncoding

from collections.abc import Mapping
from transformers.tokenization_utils import PaddingStrategy, TensorType
from transformers import (
    is_tf_available,
    is_torch_available
)
import sentencepiece as spm

from transformers.utils import logging
from transformers.utils.generic import _is_tensorflow, _is_torch

logger = logging.get_logger(__name__)

VOCAB_FILES_NAMES = {"vocab_file": "sentencepiece.bpe.model"}

PRETRAINED_VOCAB_FILES_MAP = {
    "vocab_file": {
        "indobenchmark/indobart": "https://huggingface.co/indobenchmark/indobart/resolve/main/sentencepiece.bpe.model",
        "indobenchmark/indogpt": "https://huggingface.co/indobenchmark/indogpt/resolve/main/sentencepiece.bpe.model",
        "indobenchmark/indobart-v2": "https://huggingface.co/indobenchmark/indobart-v2/resolve/main/sentencepiece.bpe.model"
    }
}

PRETRAINED_POSITIONAL_EMBEDDINGS_SIZES = {
    "indobenchmark/indobart": 768,
    "indobenchmark/indogpt": 768,
    "indobenchmark/indobart-v2": 768
}

SHARED_MODEL_IDENTIFIERS = [
    # Load with
    "indobenchmark/indobart",
    "indobenchmark/indogpt",
    "indobenchmark/indobart-v2"
]

SPIECE_UNDERLINE = "▁"

# Define type aliases and NamedTuples
TextInput = str
PreTokenizedInput = List[str]
EncodedInput = List[int]
TextInputPair = Tuple[str, str]
PreTokenizedInputPair = Tuple[List[str], List[str]]
EncodedInputPair = Tuple[List[int], List[int]]

def to_py_obj(obj):
    """
    Convert a TensorFlow tensor, PyTorch tensor, Numpy array or python list to a python list.
    """
    if isinstance(obj, (dict, UserDict)):
        return {k: to_py_obj(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_py_obj(o) for o in obj]
    elif is_tf_tensor(obj):
        return obj.numpy().tolist()
    elif is_torch_tensor(obj):
        return obj.detach().cpu().tolist()
    elif is_jax_tensor(obj):
        return np.asarray(obj).tolist()
    elif isinstance(obj, (np.ndarray, np.number)):  # tolist also works on 0d np arrays
        return obj.tolist()
    else:
        return obj

class IndoNLGTokenizer(PreTrainedTokenizer):
    vocab_files_names = VOCAB_FILES_NAMES
    pretrained_vocab_files_map = PRETRAINED_VOCAB_FILES_MAP
    max_model_input_sizes = PRETRAINED_POSITIONAL_EMBEDDINGS_SIZES
    model_input_names=['input_ids', 'attention_mask', 'decoder_input_ids', 'decoder_attention_mask', 'labels']
    input_error_message = "text input must of type `str` (single example), `List[str]` (batch of examples)."

    def __init__(
        self,
        vocab_file,
        decode_special_token=True,
        bos_token="<s>",
        eos_token="</s>",
        sep_token="</s>",
        cls_token="<s>",
        unk_token="<unk>",
        pad_token="<pad>",
        mask_token="<mask>",
        additional_special_tokens=[],
        **kwargs
    ):
        super().__init__(
            vocab_file=vocab_file,
            bos_token=bos_token,
            eos_token=eos_token,
            unk_token=unk_token,
            sep_token=sep_token,
            cls_token=cls_token,
            pad_token=pad_token,
            mask_token=mask_token,
            additional_special_tokens=additional_special_tokens,
            **kwargs,
        )
        self.sp_model = spm.SentencePieceProcessor()
        self.sp_model.Load(str(vocab_file))
        self.vocab_file = vocab_file
        self.decode_special_token = decode_special_token
        self.model_max_length = 1024

        # HACK: These tokens were added by fairseq but don't seem to be actually used when duplicated in the actual
        # sentencepiece vocabulary (this is the case for <s> and </s>
        self.special_tokens_to_ids = {
            "[javanese]": 40000,
            "[sundanese]": 40001,
            "[indonesian]": 40002,
            "<mask>": 40003
        }
        self.special_ids_to_tokens = {v: k for k, v in self.special_tokens_to_ids.items()}

        # Store Language token ID
        self.javanese_token = '[javanese]'
        self.javanese_token_id = 40000
        self.sundanese_token = '[sundanese]'
        self.sundanese_token_id = 40001
        self.indonesian_token = '[indonesian]'
        self.indonesian_token_id = 40002

        self.special_token_ids = [
            self.bos_token_id, self.eos_token_id, self.sep_token_id, self.cls_token_id,
            self.unk_token_id, self.pad_token_id, self.mask_token_id,
            self.javanese_token_id, self.sundanese_token_id, self.indonesian_token_id
        ]

    def prepare_input_for_generation(self, inputs, model_type='indobart', lang_token='[indonesian]', decoder_inputs=None,
                                             decoder_lang_token='[indonesian]', padding='longest', return_tensors=None):
        """
        Build model inputs for a specified `model_type`. There are two possible `model_type`, i.e., indobart and indogpt.

        When `model_type` is indogpt, `lang_token`, `decoder_inputs`, and `decoder_lang_token` parameters will be ignored
        and the input will be encoded in the gpt2 sequence format as follow:

        - indogpt sequence: ``<s> X``

        When `model_type` is indobart, `inputs` and `lang_token` are used as the sequence and language identifier for the indobart encoder,
        while `decoder_inputs` and `decoder_lang_token` are used as the sequence and language identifier of the decoder

        - indobart encoder sequence: ``X </s> <lang_token_id>``
        - indobart decoder sequences: ``<decoder_lang_token_id> X </s>``

        Args:
            inputs (:obj:`str` or `List[str]`):
                text sequence or list of text sequences to be tokenized.
            model_type (:obj:`str`, defaults to :obj:`indobart`):
                model type to determine the format of the tokenized sequence. Valid values are `indobart` and `indogpt`.
            lang_token (:obj:`str`, defaults to :obj:`[indonesian]`):
                language token to determine the format of the tokenized sequence. Valid values are `[indonesian]`, `[sundanese], and [javanese]`.
            decoder_inputs (:obj:`str` or `List[str]`, `optional`):
                decoder text sequence or list of text sequences to be tokenized.
            decoder_lang_token (:obj:`str`, defaults to :obj:`[indonesian]`):
                decoder language token to determine the format of the tokenized sequence. Valid values are `[indonesian]`, `[sundanese], and [javanese]`.
            padding (:obj:`str`, defaults to :obj:`longest`):
                padding strategy to pad the tokenized sequences. Valid values are `longest`, `max_length`, and `do_not_pad`.
            return_tensors (:obj:`str`, defaults to :obj:`None`):
                Returned tensor type of the tokenized sequence. When set to `None`, the return type will be List[int]. Valid values are `None`, `pt`, and `tf`

        Returns:
            :obj:`Dict`: Dictionary with `input_ids`, `attention_mask`, `decoder_input_ids` (optional), and `decoder_attention_mask` (optional)
        """
        if model_type == 'indogpt':
            # Process indogpt input
            if type(inputs) == str:
                 return self(f'<s> {inputs}', padding=padding, return_tensors=return_tensors)
            elif type(inputs) == list:
                if len(inputs) == 0 or type(inputs[0]) != str:
                    raise ValueError(IndoNLGTokenizer.input_error_message)
                else:
                    return self([f'<s> {input_data}' for input_data in inputs], padding=padding, return_tensors=return_tensors)
            else:
                raise ValueError(IndoNLGTokenizer.input_error_message)
        elif model_type == 'indobart':

            # Process encoder input
            if lang_token not in self.special_tokens_to_ids:
                raise ValueError(f"Unknown lang_token `{lang_token}`, lang_token must be either `[javanese]`, `[sundanese]`, or `[indonesian]`")
            elif type(inputs) == list:
                if len(inputs) == 0 or type(inputs[0]) != str:
                    raise ValueError(IndoNLGTokenizer.input_error_message)
            elif type(inputs) != str:
                raise ValueError(IndoNLGTokenizer.input_error_message)

            lang_id = self.special_tokens_to_ids[lang_token]
            input_batch = self(inputs, return_attention_mask=False)
            if type(inputs) == str:
                input_batch['input_ids'] = [self.bos_token_id] + input_batch['input_ids'] + [self.eos_token_id, lang_id]
            else:
                input_batch['input_ids'] = list(map(lambda input_ids: [self.bos_token_id] + input_ids + [self.eos_token_id, lang_id], input_batch['input_ids']))

            if decoder_inputs is None:
                # Return encoder input
                return self.pad(input_batch, return_tensors=return_tensors)
            else:
                # Process decoder input
                if decoder_lang_token not in self.special_tokens_to_ids:
                    raise ValueError(f"Unknown decoder_lang_token `{decoder_lang_token}`, decoder_lang_token must be either `[javanese]`, `[sundanese]`, or `[indonesian]`")
                elif type(decoder_inputs) == list:
                    if len(decoder_inputs) == 0:
                        raise ValueError(IndoNLGTokenizer.input_error_message)
                    elif type(decoder_inputs[0]) != str:
                        raise ValueError(IndoNLGTokenizer.input_error_message)
                elif type(decoder_inputs) != str:
                    raise ValueError(IndoNLGTokenizer.input_error_message)

                decoder_lang_id = self.special_tokens_to_ids[decoder_lang_token]
                decoder_input_batch = self(decoder_inputs, return_attention_mask=False)

                if type(decoder_inputs) == str:
                    labels = [self.bos_token_id] + decoder_input_batch['input_ids'] + [self.eos_token_id, decoder_lang_id]
                    decoder_input_batch['input_ids'] = [decoder_lang_id, self.bos_token_id] + decoder_input_batch['input_ids'] + [self.eos_token_id]
                else:
                    labels = list(map(lambda input_ids: [self.bos_token_id] + input_ids + [self.eos_token_id, decoder_lang_id], decoder_input_batch['input_ids']))
                    decoder_input_batch['input_ids'] = list(map(lambda input_ids: [decoder_lang_id, self.bos_token_id] + input_ids + [self.eos_token_id], decoder_input_batch['input_ids']))

                # Padding
                input_batch = self.pad(input_batch, return_tensors=return_tensors)
                decoder_input_batch = self.pad(decoder_input_batch, return_tensors=return_tensors)
                labels = self.pad({'input_ids': labels}, return_tensors=return_tensors)['input_ids']
                if not isinstance(labels, (list, tuple)):
                    labels[labels == self.pad_token_id] = -100
                else:
                    labels = list(map(lambda x: -100 if x == self.pad_token_id else x, labels))

                # Store into a single dict
                input_batch['decoder_input_ids'] = decoder_input_batch['input_ids']
                input_batch['decoder_attention_mask'] = decoder_input_batch['attention_mask']
                input_batch['labels'] = labels

                return input_batch

    def __len__(self):
        return max(self.special_ids_to_tokens) + 1

    def get_special_tokens_mask(
        self, token_ids_0: List[int], token_ids_1: Optional[List[int]] = None, already_has_special_tokens: bool = False
    ) -> List[int]:
        """
        Retrieve sequence ids from a token list that has no special tokens added. This method is called when adding
        special tokens using the tokenizer ``prepare_for_model`` method.

        Args:
            token_ids_0 (:obj:`List[int]`):
                List of IDs.
            token_ids_1 (:obj:`List[int]`, `optional`):
                Optional second list of IDs for sequence pairs.
            already_has_special_tokens (:obj:`bool`, `optional`, defaults to :obj:`False`):
                Whether or not the token list is already formatted with special tokens for the model.

        Returns:
            :obj:`List[int]`: A list of integers in the range [0, 1]: 1 for a special token, 0 for a sequence token.
        """
        if already_has_special_tokens:
            return super().get_special_tokens_mask(
                token_ids_0=token_ids_0, token_ids_1=token_ids_1, already_has_special_tokens=True
            )

        if token_ids_1 is None:
            return [1] + ([0] * len(token_ids_0)) + [1]
        return [1] + ([0] * len(token_ids_0)) + [1, 1] + ([0] * len(token_ids_1)) + [1]

    @property
    def vocab_size(self):
        return 4 + len(self.sp_model)

    def get_vocab(self):
        vocab = {self.convert_ids_to_tokens(i): i for i in range(self.vocab_size)}
        vocab.update(self.added_tokens_encoder)
        return vocab

    def _tokenize(self, text: str) -> List[str]:
        return self.sp_model.encode(text.lower(), out_type=str)

    def convert_ids_to_tokens(
        self, ids: Union[int, List[int]], skip_special_tokens: bool = False
    ) -> Union[str, List[str]]:
        """
        Converts a single index or a sequence of indices in a token or a sequence of tokens, using the vocabulary and
        added tokens.
        Args:
            ids (`int` or `List[int]`):
                The token id (or token ids) to convert to tokens.
            skip_special_tokens (`bool`, *optional*, defaults to `False`):
                Whether or not to remove special tokens in the decoding.
        Returns:
            `str` or `List[str]`: The decoded token(s).
        """
        if isinstance(ids, int):
            if ids not in self.added_tokens_decoder or ids in self.special_tokens_to_ids:
                return self._convert_id_to_token(ids, skip_special_tokens=skip_special_tokens)
            else:
                return self.added_tokens_decoder[ids]
        tokens = []
        for index in ids:
            index = int(index)
            if skip_special_tokens and index in (self.all_special_ids + list(self.special_tokens_to_ids.values())):
                continue
            if index not in self.added_tokens_decoder or index in self.special_tokens_to_ids:
                tokens.append(self._convert_id_to_token(index, skip_special_tokens=skip_special_tokens))
            else:
                tokens.append(self.added_tokens_decoder[index])
        return tokens

    def _convert_token_to_id(self, token):
        """ Converts a token (str) in an id using the vocab. """
        if token in self.special_tokens_to_ids:
            return self.special_tokens_to_ids[token]
        return self.sp_model.PieceToId(token)

    def _convert_id_to_token(self, index, skip_special_tokens=False):
        """Converts an index (integer) in a token (str) using the vocab."""
        if skip_special_tokens and index in self.special_token_ids:
            return ''

        if index in self.special_ids_to_tokens:
            return self.special_ids_to_tokens[index]

        token = self.sp_model.IdToPiece(index)
        if '<0x' in token:
            char_rep = chr(int(token[1:-1], 0))
            if char_rep.isprintable():
                return char_rep
        return token

    def __getstate__(self):
        state = self.__dict__.copy()
        state["sp_model"] = None
        return state

    def __setstate__(self, d):
        self.__dict__ = d

        # for backward compatibility
        if not hasattr(self, "sp_model_kwargs"):
            self.sp_model_kwargs = {}

        self.sp_model = spm.SentencePieceProcessor(**self.sp_model_kwargs)
        self.sp_model.Load(self.vocab_file)

    def decode(self, inputs, skip_special_tokens=False):
        outputs = super().decode(inputs, skip_special_tokens=skip_special_tokens)
        return outputs.replace(' ','').replace('▁', ' ')

    def _pad_decoder(
        self,
        encoded_inputs: Union[Dict[str, EncodedInput], BatchEncoding],
        max_length: Optional[int] = None,
        padding_strategy: PaddingStrategy = PaddingStrategy.DO_NOT_PAD,
        pad_to_multiple_of: Optional[int] = None,
        return_attention_mask: Optional[bool] = None,
    ) -> dict:
        """
        Pad encoded inputs (on left/right and up to predefined length or max length in the batch)
        Args:
            encoded_inputs:
                Dictionary of tokenized inputs (`List[int]`) or batch of tokenized inputs (`List[List[int]]`).
            max_length: maximum length of the returned list and optionally padding length (see below).
                Will truncate by taking into account the special tokens.
            padding_strategy: PaddingStrategy to use for padding.
                - PaddingStrategy.LONGEST Pad to the longest sequence in the batch
                - PaddingStrategy.MAX_LENGTH: Pad to the max length (default)
                - PaddingStrategy.DO_NOT_PAD: Do not pad
                The tokenizer padding sides are defined in self.padding_side:
                    - 'left': pads on the left of the sequences
                    - 'right': pads on the right of the sequences
            pad_to_multiple_of: (optional) Integer if set will pad the sequence to a multiple of the provided value.
                This is especially useful to enable the use of Tensor Core on NVIDIA hardware with compute capability
                >= 7.5 (Volta).
            return_attention_mask:
                (optional) Set to False to avoid returning attention mask (default: set to model specifics)
        """
        # Load from model defaults
        if return_attention_mask is None:
            return_attention_mask = "decoder_attention_mask" in self.model_input_names

        required_input = encoded_inputs[self.model_input_names[2]]

        if max_length is not None and pad_to_multiple_of is not None and (max_length % pad_to_multiple_of != 0):
            max_length = ((max_length // pad_to_multiple_of) + 1) * pad_to_multiple_of

        needs_to_be_padded = padding_strategy != PaddingStrategy.DO_NOT_PAD and len(required_input) != max_length

        # Initialize attention mask if not present.
        if return_attention_mask and "decoder_attention_mask" not in encoded_inputs:
            encoded_inputs["decoder_attention_mask"] = [1] * len(required_input)

        if needs_to_be_padded:
            difference = max_length - len(required_input)

            if self.padding_side == "right":
                if return_attention_mask:
                    encoded_inputs["decoder_attention_mask"] = encoded_inputs["decoder_attention_mask"] + [0] * difference
                if "decoder_token_type_ids" in encoded_inputs:
                    encoded_inputs["decoder_token_type_ids"] = (
                        encoded_inputs["decoder_token_type_ids"] + [self.pad_token_type_id] * difference
                    )
                if "decoder_special_tokens_mask" in encoded_inputs:
                    encoded_inputs["decoder_special_tokens_mask"] = encoded_inputs["decoder_special_tokens_mask"] + [1] * difference
                encoded_inputs[self.model_input_names[2]] = required_input + [self.pad_token_id] * difference

                label_input = encoded_inputs[self.model_input_names[4]]
                encoded_inputs[self.model_input_names[4]] = label_input + [-100] * difference
            elif self.padding_side == "left":
                if return_attention_mask:
                    encoded_inputs["decoder_attention_mask"] = [0] * difference + encoded_inputs["decoder_attention_mask"]
                if "decoder_token_type_ids" in encoded_inputs:
                    encoded_inputs["decoder_token_type_ids"] = [self.pad_token_type_id] * difference + encoded_inputs[
                        "decoder_token_type_ids"
                    ]
                if "decoder_special_tokens_mask" in encoded_inputs:
                    encoded_inputs["decoder_special_tokens_mask"] = [1] * difference + encoded_inputs["decoder_special_tokens_mask"]
                encoded_inputs[self.model_input_names[2]] = [self.pad_token_id] * difference + required_input

                label_input = encoded_inputs[self.model_input_names[4]]
                encoded_inputs[self.model_input_names[4]] = label_input + [-100] * difference
            else:
                raise ValueError("Invalid padding strategy:" + str(self.padding_side))

        return encoded_inputs

    def decode(self, inputs, skip_special_tokens=False, clean_up_tokenization_spaces: bool = True):
        outputs = super().decode(inputs, skip_special_tokens=skip_special_tokens, clean_up_tokenization_spaces=clean_up_tokenization_spaces)
        return outputs.replace(' ','').replace('▁', ' ')

    def pad(
        self,
        encoded_inputs: Union[
            BatchEncoding,
            List[BatchEncoding],
            Dict[str, EncodedInput],
            Dict[str, List[EncodedInput]],
            List[Dict[str, EncodedInput]],
        ],
        padding: Union[bool, str, PaddingStrategy] = True,
        max_length: Optional[int] = None,
        pad_to_multiple_of: Optional[int] = None,
        return_attention_mask: Optional[bool] = None,
        return_tensors: Optional[Union[str, TensorType]] = None,
        verbose: bool = True,
    ) -> BatchEncoding:
        """
        Pad a single encoded input or a batch of encoded inputs up to predefined length or to the max sequence length
        in the batch.

        Padding side (left/right) padding token ids are defined at the tokenizer level (with `self.padding_side`,
        `self.pad_token_id` and `self.pad_token_type_id`)

        <Tip>

        If the `encoded_inputs` passed are dictionary of numpy arrays, PyTorch tensors or TensorFlow tensors, the
        result will use the same type unless you provide a different tensor type with `return_tensors`. In the case of
        PyTorch tensors, you will lose the specific device of your tensors however.

        </Tip>

        Args:
            encoded_inputs ([`BatchEncoding`], list of [`BatchEncoding`], `Dict[str, List[int]]`, `Dict[str, List[List[int]]` or `List[Dict[str, List[int]]]`):
                Tokenized inputs. Can represent one input ([`BatchEncoding`] or `Dict[str, List[int]]`) or a batch of
                tokenized inputs (list of [`BatchEncoding`], *Dict[str, List[List[int]]]* or *List[Dict[str,
                List[int]]]*) so you can use this method during preprocessing as well as in a PyTorch Dataloader
                collate function.

                Instead of `List[int]` you can have tensors (numpy arrays, PyTorch tensors or TensorFlow tensors), see
                the note above for the return type.
            padding (`bool`, `str` or [`~utils.PaddingStrategy`], *optional*, defaults to `True`):
                 Select a strategy to pad the returned sequences (according to the model's padding side and padding
                 index) among:

                - `True` or `'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
                  sequence if provided).
                - `'max_length'`: Pad to a maximum length specified with the argument `max_length` or to the maximum
                  acceptable input length for the model if that argument is not provided.
                - `False` or `'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of different
                  lengths).
            max_length (`int`, *optional*):
                Maximum length of the returned list and optionally padding length (see above).
            pad_to_multiple_of (`int`, *optional*):
                If set will pad the sequence to a multiple of the provided value.

                This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability
                >= 7.5 (Volta).
            return_attention_mask (`bool`, *optional*):
                Whether to return the attention mask. If left to the default, will return the attention mask according
                to the specific tokenizer's default, defined by the `return_outputs` attribute.

                [What are attention masks?](../glossary#attention-mask)
            return_tensors (`str` or [`~utils.TensorType`], *optional*):
                If set, will return tensors instead of list of python integers. Acceptable values are:

                - `'tf'`: Return TensorFlow `tf.constant` objects.
                - `'pt'`: Return PyTorch `torch.Tensor` objects.
                - `'np'`: Return Numpy `np.ndarray` objects.
            verbose (`bool`, *optional*, defaults to `True`):
                Whether or not to print more information and warnings.
        """
        # If we have a list of dicts, let's convert it in a dict of lists
        # We do this to allow using this method as a collate_fn function in PyTorch Dataloader
        if isinstance(encoded_inputs, (list, tuple)) and isinstance(encoded_inputs[0], Mapping):
            encoded_inputs = {key: [example[key] for example in encoded_inputs] for key in encoded_inputs[0].keys()}

        # The model's main input name, usually `input_ids`, has be passed for padding
        if self.model_input_names[0] not in encoded_inputs:
            raise ValueError(
                "You should supply an encoding or a list of encodings to this method "
                f"that includes {self.model_input_names[0]}, but you provided {list(encoded_inputs.keys())}"
            )

        required_input = encoded_inputs[self.model_input_names[0]]

        if not required_input:
            if return_attention_mask:
                encoded_inputs["attention_mask"] = []
            return encoded_inputs

        # If we have PyTorch/TF/NumPy tensors/arrays as inputs, we cast them as python objects
        # and rebuild them afterwards if no return_tensors is specified
        # Note that we lose the specific device the tensor may be on for PyTorch

        first_element = required_input[0]
        if isinstance(first_element, (list, tuple)):
            # first_element might be an empty list/tuple in some edge cases so we grab the first non empty element.
            for item in required_input:
                if len(item) != 0:
                    first_element = item[0]
                    break
        # At this state, if `first_element` is still a list/tuple, it's an empty one so there is nothing to do.
        if not isinstance(first_element, (int, list, tuple)):
            if is_tf_available() and _is_tensorflow(first_element):
                return_tensors = "tf" if return_tensors is None else return_tensors
            elif is_torch_available() and _is_torch(first_element):
                return_tensors = "pt" if return_tensors is None else return_tensors
            elif isinstance(first_element, np.ndarray):
                return_tensors = "np" if return_tensors is None else return_tensors
            else:
                raise ValueError(
                    f"type of {first_element} unknown: {type(first_element)}. "
                    f"Should be one of a python, numpy, pytorch or tensorflow object."
                )

            for key, value in encoded_inputs.items():
                encoded_inputs[key] = to_py_obj(value)

        # Convert padding_strategy in PaddingStrategy
        padding_strategy, _, max_length, _ = self._get_padding_truncation_strategies(
            padding=padding, max_length=max_length, verbose=verbose
        )

        required_input = encoded_inputs[self.model_input_names[0]]
        if required_input and not isinstance(required_input[0], (list, tuple)):
            encoded_inputs = self._pad(
                encoded_inputs,
                max_length=max_length,
                padding_strategy=padding_strategy,
                pad_to_multiple_of=pad_to_multiple_of,
                return_attention_mask=return_attention_mask,
            )
            return BatchEncoding(encoded_inputs, tensor_type=return_tensors)

        batch_size = len(required_input)
        assert all(
            len(v) == batch_size for v in encoded_inputs.values()
        ), "Some items in the output dictionary have a different batch size than others."

        if padding_strategy == PaddingStrategy.LONGEST:
            max_length = max(len(inputs) for inputs in required_input)
            padding_strategy = PaddingStrategy.MAX_LENGTH

        batch_outputs = {}
        for i in range(batch_size):
            inputs = dict((k, v[i]) for k, v in encoded_inputs.items())
            outputs = self._pad(
                inputs,
                max_length=max_length,
                padding_strategy=padding_strategy,
                pad_to_multiple_of=pad_to_multiple_of,
                return_attention_mask=return_attention_mask,
            )

            # Handle decoder_input_ids
            if self.model_input_names[2] in outputs:
                max_decoder_length = max(len(inputs) for inputs in encoded_inputs[self.model_input_names[2]])
                outputs = self._pad_decoder(
                    outputs,
                    max_length=max_decoder_length,
                    padding_strategy=padding_strategy,
                    pad_to_multiple_of=pad_to_multiple_of,
                    return_attention_mask=return_attention_mask,
                )

            for key, value in outputs.items():
                if key not in batch_outputs:
                    batch_outputs[key] = []
                batch_outputs[key].append(value)

        return BatchEncoding(batch_outputs, tensor_type=return_tensors)

    def save_vocabulary(self, save_directory: str, filename_prefix: Optional[str] = None) -> Tuple[str]:
        if not os.path.isdir(save_directory):
            logger.error(f"Vocabulary path ({save_directory}) should be a directory")
            return
        out_vocab_file = os.path.join(save_directory, (filename_prefix + "-" if filename_prefix else "") + VOCAB_FILES_NAMES["vocab_file"])

        if os.path.abspath(self.vocab_file) != os.path.abspath(out_vocab_file):
            copyfile(self.vocab_file, out_vocab_file)
        return (out_vocab_file,)

## Loading datasets

In [7]:
import datasets
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=5):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [8]:
from datasets import load_dataset
import json

raw_datasets = load_dataset("joshuasiagian/indosum")

# rename kolom paragraph menjadi document
raw_datasets = raw_datasets.rename_column("paragraph", "document")

README.md: 0.00B [00:00, ?B/s]

data/train.00.jsonl:   0%|          | 0.00/36.3M [00:00<?, ?B/s]

data/dev.00.jsonl:   0%|          | 0.00/1.92M [00:00<?, ?B/s]

data/test.00.jsonl:   0%|          | 0.00/9.52M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14262 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3762 [00:00<?, ? examples/s]

### lihat dataset

In [9]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 14262
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 750
    })
    test: Dataset({
        features: ['document', 'summary'],
        num_rows: 3762
    })
})

In [10]:
raw_datasets["train"][0]

{'document': 'Jakarta, CNN Indonesia - - Dokter Ryan Thamrin, yang terkenal lewat acara Dokter Oz Indonesia, meninggal dunia pada Jumat (4 / 8) dini hari. Dokter Lula Kamal yang merupakan selebriti sekaligus rekan kerja Ryan menyebut kawannya itu sudah sakit sejak setahun yang lalu. Lula menuturkan, sakit itu membuat Ryan mesti vakum dari semua kegiatannya, termasuk menjadi pembawa acara Dokter Oz Indonesia. Kondisi itu membuat Ryan harus kembali ke kampung halamannya di Pekanbaru, Riau untuk menjalani istirahat. " Setahu saya dia orangnya sehat, tapi tahun lalu saya dengar dia sakit. (Karena) sakitnya, ia langsung pulang ke Pekanbaru, jadi kami yang mau jenguk juga susah. Barangkali mau istirahat, ya betul juga, kalau di Jakarta susah isirahatnya, " kata Lula kepada CNNIndonesia.com, Jumat (4 / 8). Lula yang mengenal Ryan sejak sebelum aktif berkarier di televisi mengaku belum sempat membesuk Ryan lantaran lokasi yang jauh. Dia juga tak tahu penyakit apa yang diderita Ryan. " Itu saya

In [11]:
show_random_elements(raw_datasets["train"])

In [12]:
show_random_elements(raw_datasets["validation"])

In [13]:
show_random_elements(raw_datasets["test"])

,document,summary
0,"Jakarta, CNN Indonesia - - Pengusaha China mengaku gerah dengan penindakan tenaga kerja asal negaranya yang kerap disebut ilegal. Timbulnya permasalahan itu dinilai bukan murni disebabkan kesalahan penanam modal. Sekretaris Jenderal China Chamber of Commerce in Indonesia (CCIC) Liu Cheng mengatakan, munculnya isu tenaga kerja ilegal adalah kesalahpahaman yang disebabkan karena perizinan. Menurutnya, selama ini izin tenaga kerja yang diberikan oleh Kementerian Ketenagarjaan (Kemenaker) untuk level manajer ke bawah hanya berlaku enam bulan. Jika ingin memperpanjang izin, maka pekerja harus registrasi ulang dan itu membutuhkan waktu sekitar dua bulan. Jika hal ini berulang terus menerus, maka biaya operasional perusahaan bisa bengkak. Selain itu, tenaga kerja harus pulang dulu ke negara asalnya sebelum memperpanjang izinnya. Tetapi di sisi lain, investor ingin agar penanaman modalnya di Indonesia bisa cepat tuntas. "" Permohonan izin kerja sementara sangat rumit dan makan waktu dua bulan, padahal izin hanya berlaku enam bulan. Akhirnya, sejumlah teknisi asal China di lapangan selalu dituduh sebagai tenaga kerja ilegal, "" ujar Liu di Hotel Borobudur, Senin (17 / 7). Tak hanya itu, kesalahpahaman juga terjadi di pemegang visa. Ia mengakui, memang banyak pekerja asal China yang memegang visa kunjungan dengan indeks Visa 211 dan 212. Namun, kedatangan mereka hanya untuk kepentingan bisnis dan bukan bekerja di Indonesia. Tapi sayangnya, golongan pemegang visa ini kerap terciduk pihak imigrasi. "" Hal ini menyulitkan kami dalam mendefinisikan perbedaan kegiatan bekerja dan kegiatan bisnis. Ada beberapa keadaan di mana pemegang visa 211 kena inspeksi di lapangan kerja, "" tuturnya. Padahal menurutnya, investor China sama sekali tak memiliki niatan untuk membawa tenaga kerja China dalam jumlah berjibun. Liu mengatakan, pengiriman tenaga kerja ke Indonesia disebabkan karena minimnya teknisi lokal yang kompeten. Jika investor mau meningkatkan kompetensi tenaga kerja lokal, maka perusahaan perlu mendidik Sumber Daya Manusia (SDM) Indonesia dalam waktu tiga hingga empat tahun. Kalau kebijakan itu ditempuh, maka perusahaan harus merogoh biaya pengembangan SDM lagi. "" Dalam proses kerja sama investasi, Indonesia sangat banyak kekurangan ahli pekerjaan di lowongan tertentu. Sehingga dalam hal ini, perusshaan China tidak pernah bermotif merekrut banyak pekerja asing untuk proyek di Indonesia, "" sebut Liu. Ia melanjutkan, perusahaan China tentu saja akan menggunakan tenaga kerja asal Indonesia karena dianggap lebih kompetitif dibanding pekerja asal negara tirai bambu itu. Sebagai buktinya, menurut data Bank Dunia, pendapatan per kapita China saat ini sudah mencapai US$8.123,2 atau lebih tinggi dibanding Indonesia yang hanya US$3.570,3. Bahkan, melihat hal itu, perusahaan China secara sukarela akan mengurangi pekerja asing dan meningkatkan proporsi pekerja lokal bagi proyeknya di Indonesia. "" Menurut teori ekonomi, perusahan asal China sama sekali tidak bermotif mendatangkan pekerjaan asing asal tiongkok untuk kelakukan pekerjaan sederhana karena biaya bekerja TKA maupun tenaga lokal juga berbeda. Perekrutan tenaga kerja Indonesia adalah yang paling ideal bagi perusahaan China, "" jelasnya. Menurut data Kementerian Ketenagakerjaan, jumlah TKA di Indonesia tahun 2016 mencapai 74.813 orang. Adapun, sebagian besar tenaga kerja ini didominasi tenaga asal China sebanyak 21.271 orang, atau sebanyak 28,43 persen dari total TKA. (gir / gir)","Pengusaha China mengaku gerah dengan penindakan tenaga kerja asal negaranya yang kerap disebut ilegal. Timbulnya permasalahan itu dinilai bukan murni disebabkan kesalahan penanam modal. Sekretaris Jenderal CCIC, Liu Cheng mengatakan, munculnya isu tenaga kerja ilegal adalah kesalahpahaman yang disebabkan karena perizinan. Selama ini izin tenaga kerja yang diberikan oleh Kemenaker untuk level manajer ke bawah hanya berlaku enam bulan."
1,"BOLASPORT.COM - CLS Knights Indonesia harus mengakui ke

## Load metrik evaluasi: rouge

In [14]:
from evaluate import load

metric = load("rouge")

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [15]:
metric

EvaluationModule(name: "rouge", module_type: "metric", features: [{'predictions': Value('string'), 'references': List(Value('string'))}, {'predictions': Value('string'), 'references': Value('string')}], usage: """
Calculates average rouge scores for a list of hypotheses and references
Args:
    predictions: list of predictions to score. Each prediction
        should be a string with tokens separated by spaces.
    references: list of reference for each prediction. Each
        reference should be a string with tokens separated by spaces.
    rouge_types: A list of rouge types to calculate.
        Valid names:
        `"rouge{n}"` (e.g. `"rouge1"`, `"rouge2"`) where: {n} is the n-gram based scoring,
        `"rougeL"`: Longest common subsequence based scoring.
        `"rougeLsum"`: rougeLsum splits text using `"
"`.
        See details in https://github.com/huggingface/datasets/issues/617
    use_stemmer: Bool indicating whether Porter stemmer should be used to strip word suffixes.
 

In [16]:
fake_preds = ["hello there", "general kenobi"]
fake_labels = ["hello there", "general kenobi"]
metric.compute(predictions=fake_preds, references=fake_labels)

{'rouge1': np.float64(1.0),
 'rouge2': np.float64(1.0),
 'rougeL': np.float64(1.0),
 'rougeLsum': np.float64(1.0)}

## Load Tokenizer

In [17]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [18]:
tokenizer = IndoNLGTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


sentencepiece.bpe.model:   0%|          | 0.00/932k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [19]:
# hanya untuk proposed model
special_tokens_dict = {'additional_special_tokens': ['<tag>']}
tokenizer.add_special_tokens(special_tokens_dict)

print(tokenizer)

IndoNLGTokenizer(name_or_path='indobenchmark/indobart-v2', vocab_size=40004, model_max_length=1024, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True), 'additional_special_tokens': ['<tag>']}, clean_up_tokenization_spaces=True)


In [20]:
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>', '<pad>', '<mask>', '<tag>']

In [21]:
print(tokenizer.all_special_ids)

[0, 2, 3, 1, 40003, 40004]


In [22]:
tokenizer("Hello, this one sentence!")

{'input_ids': [11624, 39956, 6473, 5926, 3311, 4597, 39981], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [23]:
tokenizer(["Hello, this one sentence!", "This is another sentence."])

{'input_ids': [[11624, 39956, 6473, 5926, 3311, 4597, 39981], [6473, 701, 32129, 3311, 4597, 39955]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}

In [24]:
print(tokenizer(text_target=["Hello, this one sentence!", "This is another sentence."]))

{'input_ids': [[11624, 39956, 6473, 5926, 3311, 4597, 39981], [6473, 701, 32129, 3311, 4597, 39955]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}


In [25]:
lengths = [
    len(tokenizer(doc)["input_ids"])
    for doc in raw_datasets["train"]["document"]
]

print("max:", max(lengths))
print("mean:", sum(lengths)/len(lengths))

print(">512:", sum(l > 512 for l in lengths))
print(">1024:", sum(l > 1024 for l in lengths))

Token indices sequence length is longer than the specified maximum sequence length for this model (1105 > 1024). Running this sequence through the model will result in indexing errors


max: 1589
mean: 385.1569906044033
>512: 2448
>1024: 58


### analisa panjang tokenized dokumen dan sample

In [26]:
# ==========================================
# ANALISIS PANJANG TOKEN INPUT DOCUMENT SAJA
# ==========================================

import numpy as np
import pandas as pd

# ------------------------------------------
# Hitung panjang token input document
# ------------------------------------------

document_lengths = [
    len(tokenizer(doc)["input_ids"])
    for doc in raw_datasets["train"]["document"]
]

# ------------------------------------------
# Statistik dasar
# ------------------------------------------

stats = {
    "min": np.min(document_lengths),
    "max": np.max(document_lengths),
    "mean": np.mean(document_lengths),
    "median": np.median(document_lengths),
    "p90": np.percentile(document_lengths, 90),
    "p95": np.percentile(document_lengths, 95),
    "p99": np.percentile(document_lengths, 99),
}

print("===== DOCUMENT TOKEN LENGTH STATS =====")

for k, v in stats.items():
    print(f"{k:>8}: {v:.2f}")

# ------------------------------------------
# Analisis threshold penting
# ------------------------------------------

thresholds = [256, 512, 768, 1024]

rows = []

for t in thresholds:
    count = sum(l > t for l in document_lengths)
    percent = (count / len(document_lengths)) * 100

    rows.append({
        "threshold": t,
        "count_above": count,
        "percent_above": round(percent, 2)
    })

threshold_df = pd.DataFrame(rows)

print("\n===== DOCUMENTS ABOVE THRESHOLD =====")
print(threshold_df)

# ------------------------------------------
# Optional:
# lihat document terpanjang
# ------------------------------------------

longest_idx = np.argmax(document_lengths)

print("\n===== LONGEST DOCUMENT =====")
print(f"Index        : {longest_idx}")
print(f"Token length : {document_lengths[longest_idx]}")

print("\nPreview:\n")
print(raw_datasets["train"][longest_idx]["document"][:2000])

===== DOCUMENT TOKEN LENGTH STATS =====
     min: 53.00
     max: 1589.00
    mean: 385.16
  median: 358.00
     p90: 580.00
     p95: 670.00
     p99: 882.78

===== DOCUMENTS ABOVE THRESHOLD =====
   threshold  count_above  percent_above
0        256        11706          82.08
1        512         2448          17.16
2        768          335           2.35
3       1024           58           0.41

===== LONGEST DOCUMENT =====
Index        : 2990
Token length : 1589

Preview:

Kesuksesan MSI sebagai brand gaming tak lepas dari dukungan SteelSeries. Perusahaan Denmark itu terus membantu penggarapan keyboard di notebook gaming mereka, dan dahulu, produk MSI sering dibundel bersama headset SteelSeries. Bahkan ketika sang   produsen   Taiwan mengenalkan headphone -nya sendiri (seperti DS502/DS502), MSI tetap mengadopsi rancangan gaming gear SteelSeries. Keseriusan MSI dalam meracik headset baru betul-betul terlihat di awal tahun ini, ketika sejumlah periferal gaming menjadi bagian dari p

In [27]:
# ==========================================
# ANALISIS PANJANG TOKEN SUMMARY SAJA
# ==========================================

import numpy as np
import pandas as pd

# ------------------------------------------
# Hitung panjang token summary
# ------------------------------------------

summary_lengths = [
    len(tokenizer(summary)["input_ids"])
    for summary in raw_datasets["train"]["summary"]
]

# ------------------------------------------
# Statistik dasar
# ------------------------------------------

stats = {
    "min": np.min(summary_lengths),
    "max": np.max(summary_lengths),
    "mean": np.mean(summary_lengths),
    "median": np.median(summary_lengths),
    "p90": np.percentile(summary_lengths, 90),
    "p95": np.percentile(summary_lengths, 95),
    "p99": np.percentile(summary_lengths, 99),
}

print("===== SUMMARY TOKEN LENGTH STATS =====")

for k, v in stats.items():
    print(f"{k:>8}: {v:.2f}")

# ------------------------------------------
# Analisis threshold penting
# ------------------------------------------

thresholds = [32, 64, 85, 96, 128]

rows = []

for t in thresholds:
    count = sum(l > t for l in summary_lengths)
    percent = (count / len(summary_lengths)) * 100

    rows.append({
        "threshold": t,
        "count_above": count,
        "percent_above": round(percent, 2)
    })

threshold_df = pd.DataFrame(rows)

print("\n===== SUMMARIES ABOVE THRESHOLD =====")
print(threshold_df)

# ------------------------------------------
# Optional:
# lihat summary terpanjang
# ------------------------------------------

longest_idx = np.argmax(summary_lengths)

print("\n===== LONGEST SUMMARY =====")
print(f"Index        : {longest_idx}")
print(f"Token length : {summary_lengths[longest_idx]}")

print("\nPreview:\n")
print(raw_datasets["train"][longest_idx]["summary"])

===== SUMMARY TOKEN LENGTH STATS =====
     min: 37.00
     max: 141.00
    mean: 75.06
  median: 74.00
     p90: 86.00
     p95: 90.00
     p99: 100.00

===== SUMMARIES ABOVE THRESHOLD =====
   threshold  count_above  percent_above
0         32        14262         100.00
1         64        12988          91.07
2         85         1524          10.69
3         96          253           1.77
4        128            4           0.03

===== LONGEST SUMMARY =====
Index        : 3870
Token length : 141

Preview:

Dua wakil ganda campuran Indonesia, Praveen Jordan / Debby Susanto dan Edi Subaktiar / Gloria Emanuelle Widjaja, gagal mengamankan tempat ke babak perempat final Malaysia Terbuka 2017 setelah kalah dari lawan masing-masing, Kamis (6 / 4 / 2017). Edi / Gloria kalah lebih dulu dari Lee Chun Hei Reginald / Chau Hoi Wah (Hong Kong), dengan 21 - 16, 17 - 21, 14 - 21 di Stadium Perpaduan Kuching, Sarawak. Sementara itu, Praveen / Debby ditaklukkan oleh Choi Solgyu / Chae Yoo - jung (K

In [28]:
# # ============================================
# # CONTOH PREFIX TOPIC + ANALISIS PANJANG TOKEN
# # ============================================

# # --------------------------------------------
# # SCENARIO 1
# # 10 kata dipisah spasi
# # --------------------------------------------

# scenario_1 = [
#     "<tag> ekonomi inflasi investasi pasar saham rupiah ekspor impor pajak bank <tag>",
#     "<tag> sepakbola liga pemain pelatih stadion gol suporter pertandingan transfer wasit <tag>",
#     "<tag> kesehatan rumahsakit dokter pasien vaksin penyakit operasi obat perawat klinik <tag>",
#     "<tag> pendidikan sekolah universitas mahasiswa guru kurikulum ujian penelitian kampus beasiswa <tag>",
#     "<tag> teknologi komputer internet aplikasi perangkatlunak kecerdasanbuatan data keamanan digital cloud <tag>",
#     "<tag> politik pemilu partai presiden parlemen kampanye menteri demokrasi kebijakan oposisi <tag>",
#     "<tag> musik konser penyanyi album lagu genre gitar panggung festival rekaman <tag>",
#     "<tag> olahraga bulutangkis tenis basket voli atlet turnamen medali pelatih juara <tag>",
#     "<tag> lingkungan hutan sungai limbah polusi energi iklim konservasi sampah emisi <tag>",
#     "<tag> bisnis startup perusahaan pelanggan produk pemasaran keuangan strategi penjualan inovasi <tag>",
# ]

# # --------------------------------------------
# # SCENARIO 2
# # 5 frase bigram dipisah spasi
# # --------------------------------------------

# scenario_2 = [
#     "<tag> harga minyak pasar saham sepak bola rumah sakit cuaca ekstrem <tag>",
#     "<tag> lagu pop sepak bola rumah sakit kecerdasan buatan harga pangan <tag>",
#     "<tag> vaksin covid pendidikan tinggi perdagangan internasional energi terbarukan media sosial <tag>",
#     "<tag> pemilu nasional transportasi umum keamanan siber perubahan iklim investasi asing <tag>",
#     "<tag> teknologi digital konser musik pelatihan atlet wisata lokal produk ekspor <tag>",
#     "<tag> layanan publik kecelakaan lalulintas pertandingan basket penelitian ilmiah ekonomi global <tag>",
#     "<tag> pembangunan desa berita politik pertandingan voli aplikasi mobile bursa saham <tag>",
#     "<tag> dokter spesialis kampus negeri pemain muda pelestarian hutan bisnis online <tag>",
#     "<tag> film horor musik tradisional jaringan internet pasar modal pengelolaan sampah <tag>",
#     "<tag> sistem pembayaran kesehatan mental program pemerintah olahraga nasional literasi digital <tag>",
# ]

# # --------------------------------------------
# # SCENARIO 3
# # 5 frase bigram dipisah semikolon
# # --------------------------------------------

# scenario_3 = [
#     "<tag> harga minyak ; pasar saham ; sepak bola ; rumah sakit ; cuaca ekstrem <tag>",
#     "<tag> lagu pop ; sepak bola ; rumah sakit ; kecerdasan buatan ; harga pangan <tag>",
#     "<tag> vaksin covid ; pendidikan tinggi ; perdagangan internasional ; energi terbarukan ; media sosial <tag>",
#     "<tag> pemilu nasional ; transportasi umum ; keamanan siber ; perubahan iklim ; investasi asing <tag>",
#     "<tag> teknologi digital ; konser musik ; pelatihan atlet ; wisata lokal ; produk ekspor <tag>",
#     "<tag> layanan publik ; kecelakaan lalulintas ; pertandingan basket ; penelitian ilmiah ; ekonomi global <tag>",
#     "<tag> pembangunan desa ; berita politik ; pertandingan voli ; aplikasi mobile ; bursa saham <tag>",
#     "<tag> dokter spesialis ; kampus negeri ; pemain muda ; pelestarian hutan ; bisnis online <tag>",
#     "<tag> film horor ; musik tradisional ; jaringan internet ; pasar modal ; pengelolaan sampah <tag>",
#     "<tag> sistem pembayaran ; kesehatan mental ; program pemerintah ; olahraga nasional ; literasi digital <tag>",
# ]

# # ============================================
# # FUNGSI ANALISIS TOKEN
# # ============================================

# def analyze_token_lengths(prefixes, tokenizer, scenario_name):
#     print(f"\n{'='*60}")
#     print(f"{scenario_name}")
#     print(f"{'='*60}")

#     lengths = []

#     for i, text in enumerate(prefixes):
#         tokenized = tokenizer.tokenize(text)
#         token_len = len(tokenized)

#         lengths.append(token_len)

#         print(f"\nSample {i+1}")
#         print(f"Text   : {text}")
#         print(f"Tokens : {tokenized}")
#         print(f"Length : {token_len}")

#     print(f"\n--- SUMMARY ({scenario_name}) ---")
#     print(f"Min    : {min(lengths)}")
#     print(f"Max    : {max(lengths)}")
#     print(f"Mean   : {sum(lengths)/len(lengths):.2f}")


# # ============================================
# # JALANKAN ANALISIS
# # ============================================

# analyze_token_lengths(
#     scenario_1,
#     tokenizer,
#     "SCENARIO 1 - 10 SINGLE WORDS"
# )

# analyze_token_lengths(
#     scenario_2,
#     tokenizer,
#     "SCENARIO 2 - 5 BIGRAM PHRASES"
# )

# analyze_token_lengths(
#     scenario_3,
#     tokenizer,
#     "SCENARIO 3 - 5 BIGRAM + SEMICOLON"
# )

In [29]:
# # =========================================================
# # ANALISIS THRESHOLD DENGAN RESERVED TOKEN UNTUK PREFIX
# # =========================================================

# import pandas as pd

# # ---------------------------------------------------------
# # RESERVED TOKEN
# # ---------------------------------------------------------

# reserved_tokens = 18

# print(f"Reserved tokens for proposed method: {reserved_tokens}")

# # ---------------------------------------------------------
# # Threshold yang ingin dianalisis
# # ---------------------------------------------------------

# thresholds = [256, 512, 768, 1024]

# # ---------------------------------------------------------
# # Effective threshold setelah dikurangi reserved token
# # ---------------------------------------------------------

# rows = []

# for t in thresholds:

#     effective_threshold = t - reserved_tokens

#     count_above = sum(
#         l > effective_threshold
#         for l in document_lengths
#     )

#     percent_above = (
#         count_above / len(document_lengths)
#     ) * 100

#     rows.append({
#         "original_threshold": t,
#         "reserved_tokens": reserved_tokens,
#         "effective_document_space": effective_threshold,
#         "count_above": count_above,
#         "percent_above": round(percent_above, 2)
#     })

# threshold_reserved_df = pd.DataFrame(rows)

# # ---------------------------------------------------------
# # Tampilkan hasil
# # ---------------------------------------------------------

# print("\n===== THRESHOLD ANALYSIS WITH RESERVED TOKENS =====")
# print(threshold_reserved_df)

### C-Top2Vec

mendapatkan topik

In [30]:
from top2vec import Top2Vec
import numpy as np

def build_ctop2vec_topic_texts(documents, embedding_model="all-mpnet-base-v2", top_n_words=5, separator=" "):
    model = Top2Vec(
        documents=documents,
        embedding_model=embedding_model,
        contextual_top2vec=True,
        ngram_vocab=True
    )

    topic_words, word_scores, topic_nums = model.get_topics()

    # label topik per kolom distribusi
    topic_labels_by_col = [
        separator.join(words[:top_n_words]) for words in topic_words
    ]

    topic_dist = model.get_document_topic_distribution()
    topic_rel = model.get_document_topic_relevance()

    combined = topic_dist * topic_rel
    best_topic_col = np.argmax(combined, axis=1)

    topic_texts = [topic_labels_by_col[col] for col in best_topic_col]
    return topic_texts, model



### Tokenized dataset

If you are using one of the five T5 checkpoints we have to prefix the inputs with "summarize:" (the model can also translate and it needs the prefix to know which task it has to perform).

In [33]:
if model_checkpoint in ["t5-small", "t5-base", "t5-larg", "t5-3b", "t5-11b"]:
    prefix = "summarize: "
else:
    prefix = ""

We can then write the function that will preprocess our samples. We just feed them to the `tokenizer` with the argument `truncation=True`. This will ensure that an input longer that what the model selected can handle will be truncated to the maximum length accepted by the model. The padding will be dealt with later on (in a data collator) so we pad examples to the longest length in the batch and not the whole dataset.

In [34]:
# max_input_length = 1024
max_input_length = 768 # reasonable, sudah mencover > 95% dokumen
max_target_length = 128

def preprocess_function(examples):

    # c-top2vec
    inputs = [
        f"<tag> {topic_text} <tag> " + prefix + doc
        for doc, topic_text in zip(examples["document"], examples["topic_text"])
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    ## langsung / raw
    # model_inputs = tokenizer(
    #     examples["document"],
    #     max_length=max_input_length,
    #     truncation=True
    # )


    model_inputs["labels"] = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True
    )["input_ids"]

    return model_inputs

In [ ]:
# Hanya untuk proposed model c-top2vec

import os
from datasets import load_from_disk

save_path = '/content/drive/MyDrive/s2_ctop2vec_all_mpnet_base_v2'

if os.path.exists(save_path):
    print(f"Loading dataset with topics from {save_path}...")
    raw_datasets = load_from_disk(save_path)
else:
    print("Topic dataset not found. Generating topics...")

    train_docs = [prefix + doc for doc in raw_datasets["train"]["document"]]
    train_topic_texts, train_ct2v_model = build_ctop2vec_topic_texts(train_docs, separator=" ")
    raw_datasets["train"] = raw_datasets["train"].add_column("topic_text", train_topic_texts)

    val_docs   = [prefix + doc for doc in raw_datasets["validation"]["document"]]
    val_topic_texts, val_ct2v_model = build_ctop2vec_topic_texts(val_docs, separator=" ")
    raw_datasets["validation"] = raw_datasets["validation"].add_column("topic_text", val_topic_texts)

    test_docs = [prefix + doc for doc in raw_datasets["test"]["document"]]
    test_topic_texts, test_ct2v_model = build_ctop2vec_topic_texts(test_docs, separator=" ")
    raw_datasets["test"] = raw_datasets["test"].add_column("topic_text", test_topic_texts)

    print(f"Saving dataset to {save_path}...")
    raw_datasets.save_to_disk(save_path)
    print("Done!")


Topic dataset not found. Generating topics...


2026-05-12 03:20:04,790 - top2vec - INFO - Pre-processing documents for training
INFO:top2vec:Pre-processing documents for training
/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-05-12 03:20:25,197 - top2vec - INFO - Creating vocabulary embedding
INFO:top2vec:Creating vocabulary embedding
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Embedding vocabulary:   8%|▊         | 64/803 [00:15<02:49,  4.36it/s]

In [ ]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True, batch_size=20000)

In [ ]:
tokenized_datasets

In [ ]:
tokenized_datasets["train"][0]

## Prepare model

In [ ]:
# downgrade the peft, as the new version trying to import Cache from transformers
!pip install peft==0.5.0

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# resize karena kita menambahkan `<tag>` sebelumnya di tokenizer
model.resize_token_embeddings(len(tokenizer) + len(tokenizer.added_tokens_encoder))

print(model)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

## Training & Evaluation

## Metrics

In [ ]:
import nltk
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    # Note that other metrics may not have a `use_aggregator` parameter
    # and thus will return a list, computing a metric for each sentence.
    result = metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
        use_aggregator=True,
    )
    # Extract a few results
    result = {key: value * 100 for key, value in result.items()}

    # Add mean generated length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
def print_first_predictions(results, tokenizer, num_preds=10):
    # Get the first N prediction token IDs from the results
    first_preds = results.predictions[:num_preds]

    # Ensure no -100 are in the predictions (in case padding was ignored)
    first_preds = np.where(first_preds != -100, first_preds, tokenizer.pad_token_id)

    # Decode the token IDs back into readable text
    decoded_preds = tokenizer.batch_decode(first_preds, skip_special_tokens=True)

    # Print the results
    for i, pred in enumerate(decoded_preds):
        print(f"Prediction {i+1}:\n{pred}\n")

### Training

In [ ]:
model_name = model_checkpoint.split("/")[-1]

In [ ]:
# hyperparameter - finetuning
batch_size = 8
learning_rate = 3.75e-5
weight_decay = 0.01
num_train_epochs = 3

In [ ]:
# args = Seq2SeqTrainingArguments(
#     f"{model_name}-finetuned-indosum",
#     evaluation_strategy="epoch",
#     # save_strategy="epoch",  # penting
#     save_strategy = "no",
#     learning_rate=learning_rate,
#     per_device_train_batch_size=batch_size,
#     per_device_eval_batch_size=batch_size,
#     weight_decay=weight_decay,
#     num_train_epochs=num_train_epochs,
#     predict_with_generate=True,
#     # generation_num_beams=4,  # tambahan
#     fp16=True,
#     push_to_hub=False,
#     # save_total_limit=2,  # aman
#     save_total_limit=-1,
#     # load_best_model_at_end=True,
#     # metric_for_best_model="rougeL",
#     # greater_is_better=True,
#     generation_max_length=85,
# )

In [ ]:
# trainer = Seq2SeqTrainer(
#     model,
#     args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["validation"],
#     data_collator=data_collator,
#     tokenizer=tokenizer,
#     compute_metrics=compute_metrics
# )

In [ ]:
# trainer.train()

### Evaluasi (3 split)

In [ ]:
# # Increase eval batch size for faster inference (without sacrificing quality)
# trainer.args.per_device_eval_batch_size = 32

In [ ]:
# train_results = trainer.predict(tokenized_datasets["train"])

In [ ]:
# print(train_results.metrics)

In [ ]:
# print_first_predictions(train_results, tokenizer)

In [ ]:
# val_results = trainer.predict(tokenized_datasets["validation"])

In [ ]:
# print(val_results.metrics)

In [ ]:
# print_first_predictions(val_results, tokenizer)

In [ ]:
# test_results = trainer.predict(tokenized_datasets["test"])

In [ ]:
# print(test_results.metrics)

In [ ]:
# print_first_predictions(test_results, tokenizer)

## List dependency

In [ ]:
!pip list


# Multi-Experiment Fine-Tuning

Cell berikut digunakan untuk menjalankan beberapa kombinasi eksperimen:
- max_generation_length: 60, 85, 100
- learning_rate: 3.75e-5 dan 5e-5

Tujuan:
- mempermudah hyperparameter comparison
- menghasilkan tabel eksperimen otomatis
- mendukung ablation/sensitivity analysis pada thesis


In [ ]:
from transformers import set_seed
import pandas as pd
import gc
import torch
import os
import json

set_seed(42)

BASE_DIR = "/content/drive/MyDrive/s2_proposedmodel"
os.makedirs(BASE_DIR, exist_ok=True)

experiment_configs = [
    {"max_gen_len": 60,  "lr": 3.75e-5},
    {"max_gen_len": 60,  "lr": 5e-5},

    {"max_gen_len": 85,  "lr": 3.75e-5},
    {"max_gen_len": 85,  "lr": 5e-5},

    {"max_gen_len": 100, "lr": 3.75e-5},
    {"max_gen_len": 100, "lr": 5e-5},
]

all_results = []

for exp in experiment_configs:

    experiment_name = f"len{exp['max_gen_len']}_lr{exp['lr']}"

    result_file = os.path.join(
        BASE_DIR,
        f"{experiment_name}_done.json"
    )

    # skip jika sudah selesai
    if os.path.exists(result_file):
      with open(result_file, "r") as f:
          saved_result = json.load(f)
      print("=" * 70)
      print(f"SKIP: {experiment_name} sudah selesai")
      print("HASIL VALIDATION:")

      for key, value in saved_result.items():
          print(f"{key}: {value}")

      print("=" * 70)

      all_results.append(saved_result)
      continue

    print("=" * 70)
    print(f"TRAINING EXPERIMENT | {experiment_name}")
    print("=" * 70)

    # reload fresh model tiap eksperimen
    model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

    training_args = Seq2SeqTrainingArguments(

        # checkpoint temporary di runtime
        output_dir=f"./temp_{experiment_name}",

        evaluation_strategy="epoch",

        # hemat storage
        save_strategy="no",

        learning_rate=exp["lr"],

        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        weight_decay=0.01,

        num_train_epochs=3,

        predict_with_generate=True,

        fp16=True,

        # logging_steps=100,

        load_best_model_at_end=False,

        generation_max_length=exp["max_gen_len"],
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    train_logs = trainer.state.log_history
    training_loss = None
    for log in reversed(train_logs):
        if "loss" in log:
            training_loss = log["loss"]
            break

    # validation evaluation ONLY
    val_metrics = trainer.evaluate(
        eval_dataset=tokenized_datasets["validation"],
        metric_key_prefix="validation",
        # max_length=exp["max_gen_len"]
    )

    result_row = {
        "experiment_name": experiment_name,
        "max_gen_len": exp["max_gen_len"],
        "learning_rate": exp["lr"],
        "training_loss": training_loss,
        **val_metrics
    }

    all_results.append(result_row)

    # save result per eksperimen
    with open(result_file, "w") as f:
        json.dump(result_row, f)

    # update csv global
    results_df = pd.DataFrame(all_results)

    results_df = results_df.sort_values(
        by=["validation_rougeLsum", "validation_rouge2"],
        ascending=False
    )

    csv_path = os.path.join(BASE_DIR, "all_results.csv")
    results_df.to_csv(csv_path, index=False)

    print(f"RESULT SAVED: {result_file}")

    # clear memory
    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()

# final dataframe
results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by=["validation_rougeLsum", "validation_rouge2"],
    ascending=False
)

results_df


In [ ]:
# clear memory
# del trainer
# del model
# gc.collect()
# torch.cuda.empty_cache()

exp = {"max_gen_len": 85, "lr": 3.75e-5}
experiment_name = f"test_len{exp['max_gen_len']}_lr{exp['lr']}"

print("=" * 70)
print(f"TRAINING & TEST EVALUATION | {experiment_name}")
print("=" * 70)

# reload fresh model
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

training_args = Seq2SeqTrainingArguments(
    output_dir=f"./temp_{experiment_name}",
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=exp["lr"],
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    # logging_steps=100,
    load_best_model_at_end=False,
    generation_max_length=exp["max_gen_len"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Lakukan training terlebih dahulu karena kita tidak menyimpan checkpoint
trainer.train()

train_logs = trainer.state.log_history
training_loss = None
for log in reversed(train_logs):
    if "loss" in log:
        training_loss = log["loss"]
        break

# Evaluasi menggunakan dataset test
test_metrics = trainer.evaluate(
    eval_dataset=tokenized_datasets["test"],
    metric_key_prefix="test",
    # max_length=exp["max_gen_len"]
)

print("\n" + "=" * 70)
print("HASIL TEST:")
for key, value in test_metrics.items():
    print(f"{key}: {value}")

# print training loss
print(f"training_loss: {training_loss}")

print("=" * 70)
